<a href="https://colab.research.google.com/github/Haya746/NLP/blob/main/NLP_Lab_Mega_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Lab — Consolidated Mega Notebook

This notebook merges every unique concept from Assignments 1, 2 and 3 into one place. Each code cell covers exactly **one topic**, and starts with the library import(s) that topic needs, followed by the code for it — so every cell can be read (and run) on its own.

Two consistent corpora are used throughout instead of switching datasets every cell:

- **Corpus A** — one short multi-sentence paragraph, reused for every text-structure task: tokenization, cleaning, HTML/URL removal, stopword removal, n-grams, stemming, lemmatization, POS tagging, NER, Bag of Words, TF-IDF.
- **Corpus B** — one small labelled set of product reviews, reused for every sentiment/classification task: TextBlob, VADER, their comparison, and Naive Bayes classification.

## Part A — Text Preprocessing & Structural NLP (Corpus A)

In [ ]:
# ---- CORPUS A (used for every task in Part A) ----
corpus_a = """Artificial Intelligence is transforming healthcare and education.
OpenAI developed ChatGPT in San Francisco.
Dr. Sarah Lee teaches Machine Learning at Stanford University.
Google launched Gemini in 2024.
Visit https://openai.com for more details!!!
<p>Machine Learning improves healthcare rapidly.</p>"""

print(corpus_a)

Artificial Intelligence is transforming healthcare and education.
OpenAI developed ChatGPT in San Francisco.
Dr. Sarah Lee teaches Machine Learning at Stanford University.
Google launched Gemini in 2024.
Visit https://openai.com for more details!!!
<p>Machine Learning improves healthcare rapidly.</p>


In [ ]:
# ---- TOKENIZATION ----
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

tokens = word_tokenize(corpus_a.lower())
print("Tokens:")
print(tokens)

Tokens:
['artificial', 'intelligence', 'is', 'transforming', 'healthcare', 'and', 'education', '.', 'openai', 'developed', 'chatgpt', 'in', 'san', 'francisco', '.', 'dr.', 'sarah', 'lee', 'teaches', 'machine', 'learning', 'at', 'stanford', 'university', '.', 'google', 'launched', 'gemini', 'in', '2024.', 'visit', 'https', ':', '//openai.com', 'for', 'more', 'details', '!', '!', '!', '<', 'p', '>', 'machine', 'learning', 'improves', 'healthcare', 'rapidly.', '<', '/p', '>']


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# ---- TEXT CLEANING: LOWERCASE, REMOVE PUNCTUATION, REMOVE NUMBERS ----
import re

lower_text = corpus_a.lower()
print("Lowercase:\n", lower_text)

no_punct = re.sub(r'[^\w\s]', '', lower_text)
print("\nWithout Punctuation:\n", no_punct)

no_numbers = re.sub(r'\d+', '', no_punct)
print("\nWithout Numbers:\n", no_numbers)

Lowercase:
 artificial intelligence is transforming healthcare and education.
openai developed chatgpt in san francisco.
dr. sarah lee teaches machine learning at stanford university.
google launched gemini in 2024.
visit https://openai.com for more details!!!
<p>machine learning improves healthcare rapidly.</p>

Without Punctuation:
 artificial intelligence is transforming healthcare and education
openai developed chatgpt in san francisco
dr sarah lee teaches machine learning at stanford university
google launched gemini in 2024
visit httpsopenaicom for more details
pmachine learning improves healthcare rapidlyp

Without Numbers:
 artificial intelligence is transforming healthcare and education
openai developed chatgpt in san francisco
dr sarah lee teaches machine learning at stanford university
google launched gemini in 
visit httpsopenaicom for more details
pmachine learning improves healthcare rapidlyp


In [ ]:
# ---- HTML TAG & URL REMOVAL ----
import re

text_with_markup = corpus_a.lower()
no_html = re.sub(r'<.*?>', '', text_with_markup)
no_url = re.sub(r'http\S+|www\S+', '', no_html)
cleaned = re.sub(r'[^a-zA-Z\s]', '', no_url).strip()

print("After removing HTML tags:\n", no_html)
print("\nAfter removing URLs:\n", no_url)
print("\nFully cleaned (letters only):\n", cleaned)

After removing HTML tags:
 artificial intelligence is transforming healthcare and education.
openai developed chatgpt in san francisco.
dr. sarah lee teaches machine learning at stanford university.
google launched gemini in 2024.
visit https://openai.com for more details!!!
machine learning improves healthcare rapidly.

After removing URLs:
 artificial intelligence is transforming healthcare and education.
openai developed chatgpt in san francisco.
dr. sarah lee teaches machine learning at stanford university.
google launched gemini in 2024.
visit  for more details!!!
machine learning improves healthcare rapidly.

Fully cleaned (letters only):
 artificial intelligence is transforming healthcare and education
openai developed chatgpt in san francisco
dr sarah lee teaches machine learning at stanford university
google launched gemini in 
visit  for more details
machine learning improves healthcare rapidly


In [ ]:
# ---- STOPWORD REMOVAL ----
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')

tokens_clean = word_tokenize(cleaned)
stop_words = set(stopwords.words('english'))
filtered_tokens = [w for w in tokens_clean if w not in stop_words]

print("Tokens before stopword removal:")
print(tokens_clean)
print("\nTokens after stopword removal:")
print(filtered_tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hayasachin/nltk_data...


Tokens before stopword removal:
['artificial', 'intelligence', 'is', 'transforming', 'healthcare', 'and', 'education', 'openai', 'developed', 'chatgpt', 'in', 'san', 'francisco', 'dr', 'sarah', 'lee', 'teaches', 'machine', 'learning', 'at', 'stanford', 'university', 'google', 'launched', 'gemini', 'in', 'visit', 'for', 'more', 'details', 'machine', 'learning', 'improves', 'healthcare', 'rapidly']

Tokens after stopword removal:
['artificial', 'intelligence', 'transforming', 'healthcare', 'education', 'openai', 'developed', 'chatgpt', 'san', 'francisco', 'dr', 'sarah', 'lee', 'teaches', 'machine', 'learning', 'stanford', 'university', 'google', 'launched', 'gemini', 'visit', 'details', 'machine', 'learning', 'improves', 'healthcare', 'rapidly']


[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# ---- N-GRAM MODELING (UNIGRAM, BIGRAM, TRIGRAM) ----
from collections import Counter
from nltk.util import ngrams
from nltk.tokenize import word_tokenize

ngram_tokens = word_tokenize(corpus_a.lower())

unigrams = list(ngrams(ngram_tokens, 1))
bigrams = list(ngrams(ngram_tokens, 2))
trigrams = list(ngrams(ngram_tokens, 3))

print("Unigrams:", unigrams)
print("\nBigrams:", bigrams)
print("\nTrigrams:", trigrams)

uni_freq = Counter(unigrams)
bi_freq = Counter(bigrams)
tri_freq = Counter(trigrams)

print("\nUnigram Frequency:", uni_freq)
print("\nBigram Frequency:", bi_freq)
print("\nTrigram Frequency:", tri_freq)

# Probabilities
total_uni = len(unigrams)
print("\nUnigram Probabilities:")
for w, c in uni_freq.items():
    print(w[0], "=", round(c/total_uni, 3))

total_bi = len(bigrams)
print("\nBigram Probabilities:")
for w, c in bi_freq.items():
    print(w, "=", round(c/total_bi, 3))

total_tri = len(trigrams)
print("\nTrigram Probabilities:")
for w, c in tri_freq.items():
    print(w, "=", round(c/total_tri, 3))

# Most frequent n-grams
for name, freq in [("Unigram", uni_freq), ("Bigram", bi_freq), ("Trigram", tri_freq)]:
    max_count = max(freq.values())
    most_common = [item for item, c in freq.items() if c == max_count]
    print(f"\nMost frequent {name} (freq={max_count}):")
    for item in most_common:
        print(" ->", " ".join(item))

Unigrams: [('artificial',), ('intelligence',), ('is',), ('transforming',), ('healthcare',), ('and',), ('education',), ('.',), ('openai',), ('developed',), ('chatgpt',), ('in',), ('san',), ('francisco',), ('.',), ('dr.',), ('sarah',), ('lee',), ('teaches',), ('machine',), ('learning',), ('at',), ('stanford',), ('university',), ('.',), ('google',), ('launched',), ('gemini',), ('in',), ('2024.',), ('visit',), ('https',), (':',), ('//openai.com',), ('for',), ('more',), ('details',), ('!',), ('!',), ('!',), ('<',), ('p',), ('>',), ('machine',), ('learning',), ('improves',), ('healthcare',), ('rapidly.',), ('<',), ('/p',), ('>',)]

Bigrams: [('artificial', 'intelligence'), ('intelligence', 'is'), ('is', 'transforming'), ('transforming', 'healthcare'), ('healthcare', 'and'), ('and', 'education'), ('education', '.'), ('.', 'openai'), ('openai', 'developed'), ('developed', 'chatgpt'), ('chatgpt', 'in'), ('in', 'san'), ('san', 'francisco'), ('francisco', '.'), ('.', 'dr.'), ('dr.', 'sarah'), ('s

In [ ]:
# ---- STEMMING: PORTER & SNOWBALL ----
from nltk.stem import PorterStemmer, SnowballStemmer

stem_tokens = filtered_tokens  # reuse cleaned, stopword-free tokens from Corpus A

porter = PorterStemmer()
porter_output = [porter.stem(w) for w in stem_tokens]

snowball = SnowballStemmer("english")
snowball_output = [snowball.stem(w) for w in stem_tokens]

print("Porter Stemmer:")
print(porter_output)
print("\nSnowball Stemmer:")
print(snowball_output)

Porter Stemmer:
['artifici', 'intellig', 'transform', 'healthcar', 'educ', 'openai', 'develop', 'chatgpt', 'san', 'francisco', 'dr', 'sarah', 'lee', 'teach', 'machin', 'learn', 'stanford', 'univers', 'googl', 'launch', 'gemini', 'visit', 'detail', 'machin', 'learn', 'improv', 'healthcar', 'rapidli']

Snowball Stemmer:
['artifici', 'intellig', 'transform', 'healthcar', 'educ', 'openai', 'develop', 'chatgpt', 'san', 'francisco', 'dr', 'sarah', 'lee', 'teach', 'machin', 'learn', 'stanford', 'univers', 'googl', 'launch', 'gemini', 'visit', 'detail', 'machin', 'learn', 'improv', 'healthcar', 'rapid']


In [ ]:
# ---- LEMMATIZATION (spaCy) & COMPARISON WITH STEMMERS ----
import spacy
nlp = spacy.load("en_core_web_sm")

doc_for_lemma = nlp(" ".join(filtered_tokens))
lemma_output = [token.lemma_ for token in doc_for_lemma if not token.is_punct]

print("spaCy Lemmatizer:")
print(lemma_output)

print("\nComparison — Token vs Porter vs Snowball vs Lemma:")
print(f"{'Token':<15}{'Porter':<15}{'Snowball':<15}{'Lemma'}")
for i in range(len(filtered_tokens)):
    p = porter_output[i] if i < len(porter_output) else "-"
    s = snowball_output[i] if i < len(snowball_output) else "-"
    l = lemma_output[i] if i < len(lemma_output) else "-"
    print(f"{filtered_tokens[i]:<15}{p:<15}{s:<15}{l}")

spaCy Lemmatizer:
['artificial', 'intelligence', 'transform', 'healthcare', 'education', 'openai', 'develop', 'chatgpt', 'san', 'francisco', 'dr', 'sarah', 'lee', 'teach', 'machine', 'learn', 'stanford', 'university', 'google', 'launch', 'gemini', 'visit', 'detail', 'machine', 'learning', 'improve', 'healthcare', 'rapidly']

Comparison — Token vs Porter vs Snowball vs Lemma:
Token          Porter         Snowball       Lemma
artificial     artifici       artifici       artificial
intelligence   intellig       intellig       intelligence
transforming   transform      transform      transform
healthcare     healthcar      healthcar      healthcare
education      educ           educ           education
openai         openai         openai         openai
developed      develop        develop        develop
chatgpt        chatgpt        chatgpt        chatgpt
san            san            san            san
francisco      francisco      francisco      francisco
dr             dr            

In [ ]:
# ---- POS TAGGING (spaCy) ----
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")
doc = nlp(corpus_a)

print("--- POS Tagging ---")
for token in doc:
    print(f"{token.text:<15} | {token.pos_}")

pos_counts = Counter([token.pos_ for token in doc])
total_nouns = pos_counts.get("NOUN", 0) + pos_counts.get("PROPN", 0)
total_verbs = pos_counts.get("VERB", 0)
print("\nTotal Nouns:", total_nouns)
print("Total Verbs:", total_verbs)

--- POS Tagging ---
Artificial      | PROPN
Intelligence    | PROPN
is              | AUX
transforming    | VERB
healthcare      | NOUN
and             | CCONJ
education       | NOUN
.               | PUNCT

               | SPACE
OpenAI          | PROPN
developed       | VERB
ChatGPT         | NOUN
in              | ADP
San             | PROPN
Francisco       | PROPN
.               | PUNCT

               | SPACE
Dr.             | PROPN
Sarah           | PROPN
Lee             | PROPN
teaches         | VERB
Machine         | PROPN
Learning        | PROPN
at              | ADP
Stanford        | PROPN
University      | PROPN
.               | PUNCT

               | SPACE
Google          | PROPN
launched        | VERB
Gemini          | PROPN
in              | ADP
2024            | NUM
.               | PUNCT

               | SPACE
Visit           | VERB
https://openai.com | X
for             | ADP
more            | ADJ
details         | NOUN
!               | PUNCT
!               | PU

In [ ]:
# ---- NAMED ENTITY RECOGNITION (spaCy) ----
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(corpus_a)

print("--- Named Entities ---")
for ent in doc.ents:
    print(f"{ent.text:<25} | {ent.label_}")

print("\nTotal Named Entities:", len(doc.ents))

person_ents = list(set(e.text for e in doc.ents if e.label_ == "PERSON"))
org_ents = list(set(e.text for e in doc.ents if e.label_ == "ORG"))
gpe_ents = list(set(e.text for e in doc.ents if e.label_ == "GPE"))
date_ents = list(set(e.text for e in doc.ents if e.label_ == "DATE"))

print("\nPERSON:", person_ents)
print("ORG:", org_ents)
print("GPE:", gpe_ents)
print("DATE:", date_ents)

--- Named Entities ---
Artificial Intelligence   | PERSON
San Francisco             | GPE
Sarah Lee                 | PERSON
Machine Learning          | PERSON
Stanford University       | ORG
Google                    | ORG
Gemini                    | ORG
2024                      | DATE

Total Named Entities: 8

PERSON: ['Sarah Lee', 'Artificial Intelligence', 'Machine Learning']
ORG: ['Stanford University', 'Gemini', 'Google']
GPE: ['San Francisco']
DATE: ['2024']


In [ ]:
# ---- BAG OF WORDS (CountVectorizer) ----
from sklearn.feature_extraction.text import CountVectorizer

# Split Corpus A into sentence-level documents
documents = [s.strip() for s in corpus_a.replace("\n", " ").split(".") if s.strip()]

cv = CountVectorizer(stop_words='english')
bow_matrix = cv.fit_transform(documents)

print("Documents:")
for d in documents:
    print(" -", d)

print("\nVocabulary:")
print(cv.get_feature_names_out())

print("\nBoW Matrix:")
print(bow_matrix.toarray())

Documents:
 - Artificial Intelligence is transforming healthcare and education
 - OpenAI developed ChatGPT in San Francisco
 - Dr
 - Sarah Lee teaches Machine Learning at Stanford University
 - Google launched Gemini in 2024
 - Visit https://openai
 - com for more details!!! <p>Machine Learning improves healthcare rapidly
 - </p>

Vocabulary:
['2024' 'artificial' 'chatgpt' 'com' 'details' 'developed' 'dr'
 'education' 'francisco' 'gemini' 'google' 'healthcare' 'https' 'improves'
 'intelligence' 'launched' 'learning' 'lee' 'machine' 'openai' 'rapidly'
 'san' 'sarah' 'stanford' 'teaches' 'transforming' 'university' 'visit']

BoW Matrix:
[[0 1 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 1 1 0 1 0]
 [1 0 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1]
 [0

In [ ]:
# ---- TF-IDF VECTORIZATION ----
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(documents)

print("TF-IDF Vocabulary:")
print(tfidf.get_feature_names_out())

print("\nTF-IDF Matrix:")
print(tfidf_matrix.toarray())

TF-IDF Vocabulary:
['2024' 'artificial' 'chatgpt' 'com' 'details' 'developed' 'dr'
 'education' 'francisco' 'gemini' 'google' 'healthcare' 'https' 'improves'
 'intelligence' 'launched' 'learning' 'lee' 'machine' 'openai' 'rapidly'
 'san' 'sarah' 'stanford' 'teaches' 'transforming' 'university' 'visit']

TF-IDF Matrix:
[[0.         0.46114911 0.         0.         0.         0.
  0.         0.46114911 0.         0.         0.         0.38647895
  0.         0.         0.46114911 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.46114911 0.         0.        ]
 [0.         0.         0.46114911 0.         0.         0.46114911
  0.         0.         0.46114911 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.38647895 0.         0.46114911 0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  1.         0.         0.    

## Part B — Sentiment Analysis & Text Classification (Corpus B)

In [ ]:
# ---- CORPUS B (used for every task in Part B) ----
reviews = [
    "I love this phone, the camera is excellent.",
    "This laptop is terrible and painfully slow.",
    "Amazing display and outstanding build quality!",
    "Worst customer service I have ever experienced.",
    "Good value for money, highly recommended.",
    "Poor performance, very disappointing overall.",
    "Excellent sound quality and elegant design.",
    "Battery drains too fast, quite frustrating."
]

labels = [
    "Positive", "Negative", "Positive", "Negative",
    "Positive", "Negative", "Positive", "Negative"
]

for r, l in zip(reviews, labels):
    print(f"{l:<10} -> {r}")

Positive   -> I love this phone, the camera is excellent.
Negative   -> This laptop is terrible and painfully slow.
Positive   -> Amazing display and outstanding build quality!
Negative   -> Worst customer service I have ever experienced.
Positive   -> Good value for money, highly recommended.
Negative   -> Poor performance, very disappointing overall.
Positive   -> Excellent sound quality and elegant design.
Negative   -> Battery drains too fast, quite frustrating.


In [ ]:
# ---- SENTIMENT ANALYSIS: TEXTBLOB ----
from textblob import TextBlob

positive = negative = neutral = 0
for review in reviews:
    blob = TextBlob(review)
    polarity = blob.sentiment.polarity
    if polarity > 0:
        sentiment = "Positive"
        positive += 1
    elif polarity < 0:
        sentiment = "Negative"
        negative += 1
    else:
        sentiment = "Neutral"
        neutral += 1
    print("Review     :", review)
    print("Polarity   :", polarity)
    print("Sentiment  :", sentiment)
    print("-" * 40)

print("Total Positive:", positive)
print("Total Negative:", negative)
print("Total Neutral :", neutral)

Review     : I love this phone, the camera is excellent.
Polarity   : 0.75
Sentiment  : Positive
----------------------------------------
Review     : This laptop is terrible and painfully slow.
Polarity   : -0.65
Sentiment  : Negative
----------------------------------------
Review     : Amazing display and outstanding build quality!
Polarity   : 0.6125
Sentiment  : Positive
----------------------------------------
Review     : Worst customer service I have ever experienced.
Polarity   : -0.09999999999999998
Sentiment  : Negative
----------------------------------------
Review     : Good value for money, highly recommended.
Polarity   : 0.43
Sentiment  : Positive
----------------------------------------
Review     : Poor performance, very disappointing overall.
Polarity   : -0.39333333333333337
Sentiment  : Negative
----------------------------------------
Review     : Excellent sound quality and elegant design.
Polarity   : 0.6333333333333333
Sentiment  : Positive
-------------------

In [ ]:
# ---- SENTIMENT ANALYSIS: VADER ----
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
for review in reviews:
    score = analyzer.polarity_scores(review)
    if score["compound"] >= 0.05:
        sentiment = "Positive"
    elif score["compound"] <= -0.05:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"
    print("Review         :", review)
    print("Negative Score :", score["neg"])
    print("Neutral Score  :", score["neu"])
    print("Positive Score :", score["pos"])
    print("Compound Score :", score["compound"])
    print("Sentiment      :", sentiment)
    print("-" * 40)

Review         : I love this phone, the camera is excellent.
Negative Score : 0.0
Neutral Score  : 0.432
Positive Score : 0.568
Compound Score : 0.836
Sentiment      : Positive
----------------------------------------
Review         : This laptop is terrible and painfully slow.
Negative Score : 0.565
Neutral Score  : 0.435
Positive Score : 0.0
Compound Score : -0.7579
Sentiment      : Negative
----------------------------------------
Review         : Amazing display and outstanding build quality!
Negative Score : 0.0
Neutral Score  : 0.331
Positive Score : 0.669
Compound Score : 0.8439
Sentiment      : Positive
----------------------------------------
Review         : Worst customer service I have ever experienced.
Negative Score : 0.406
Neutral Score  : 0.594
Positive Score : 0.0
Compound Score : -0.6249
Sentiment      : Negative
----------------------------------------
Review         : Good value for money, highly recommended.
Negative Score : 0.0
Neutral Score  : 0.289
Positive Scor

In [ ]:
# ---- COMPARISON: TEXTBLOB vs VADER ----
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
for review in reviews:
    blob = TextBlob(review)
    tb_polarity = blob.sentiment.polarity
    tb_sentiment = "Positive" if tb_polarity > 0 else "Negative" if tb_polarity < 0 else "Neutral"

    score = analyzer.polarity_scores(review)
    vd_sentiment = "Positive" if score["compound"] >= 0.05 else "Negative" if score["compound"] <= -0.05 else "Neutral"

    print("Review  :", review)
    print(f"TextBlob : {tb_sentiment} | Polarity = {tb_polarity}")
    print(f"VADER    : {vd_sentiment} | Compound = {score['compound']}")
    print("-" * 50)

Review  : I love this phone, the camera is excellent.
TextBlob : Positive | Polarity = 0.75
VADER    : Positive | Compound = 0.836
--------------------------------------------------
Review  : This laptop is terrible and painfully slow.
TextBlob : Negative | Polarity = -0.65
VADER    : Negative | Compound = -0.7579
--------------------------------------------------
Review  : Amazing display and outstanding build quality!
TextBlob : Positive | Polarity = 0.6125
VADER    : Positive | Compound = 0.8439
--------------------------------------------------
Review  : Worst customer service I have ever experienced.
TextBlob : Negative | Polarity = -0.09999999999999998
VADER    : Negative | Compound = -0.6249
--------------------------------------------------
Review  : Good value for money, highly recommended.
TextBlob : Positive | Polarity = 0.43
VADER    : Positive | Compound = 0.7501
--------------------------------------------------
Review  : Poor performance, very disappointing overall.
Text

In [ ]:
# ---- TEXT CLASSIFICATION: TF-IDF + MULTINOMIAL NAIVE BAYES (fit on all data, predict new text) ----
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

df = pd.DataFrame({"text": reviews, "label": labels})

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["text"])
y = df["label"]

model = MultinomialNB()
model.fit(X, y)

new_text = ["Excellent battery and great camera"]
new_vector = vectorizer.transform(new_text)
prediction = model.predict(new_vector)

print("Sentence           :", new_text[0])
print("Predicted Sentiment:", prediction[0])

Sentence           : Excellent battery and great camera
Predicted Sentiment: Positive


In [ ]:
# ---- TEXT CLASSIFICATION WITH TRAIN-TEST SPLIT + ACCURACY ----
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

df = pd.DataFrame({"text": reviews, "label": labels})

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.25, random_state=42
)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)

print("Actual Labels    :", list(y_test))
print("Predicted Labels :", list(predictions))
print("Accuracy         :", accuracy_score(y_test, predictions))

Actual Labels    : ['Negative', 'Negative']
Predicted Labels : [np.str_('Positive'), np.str_('Positive')]
Accuracy         : 0.0
